<a href="https://colab.research.google.com/github/Shekinah-Ineza/formative3-group19/blob/main/Formative3_Part2_Bayes_Team19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Formative 3, Part 2: Bayesian Probability on IMDb Reviews Team 19

  |Teammates: MUTONI KEIRA  Habibllah Ayoola Ayodele Merveille Shekina Ineza
  
We apply Bayes' theorem to estimate how likely a review is positive when it contains a given keyword, using the IMDb 50K movie reviews dataset and basic Python only.

We chose Excellent, Wonderful and Amazing for positive words. And we chose Boring, Terrible and Waste for negative words.

In [20]:
import csv

def load_reviews(path):
    """Read the CSV and return a list of (review_text, sentiment) pairs."""
    reviews = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            reviews.append((row["review"], row["sentiment"]))
    return reviews

reviews = load_reviews("/content/IMDB Dataset.csv")
print("Total reviews:", len(reviews))

Total reviews: 50000


In [21]:
def tokenize(text):
    """Lowercase, remove HTML breaks and punctuation, return the set of words."""
    text = text.lower().replace("<br />", " ")
    for ch in '.,!?;:"()[]-/':
        text = text.replace(ch, " ")
    return set(text.split())

def prepare(reviews):
    """Return a list of (word_set, is_positive) pairs."""
    return [(tokenize(text), label == "positive") for text, label in reviews]

data = prepare(reviews)
print("Prepared:", len(data))
print("Is 'excellent' in the first review?", "excellent" in data[0][0])

Prepared: 50000
Is 'excellent' in the first review? False


In [25]:
def bayes_for_keyword(data, keyword):
    """Compute prior, likelihood, marginal and posterior P(Positive | keyword)."""
    total = len(data)
    n_pos = sum(1 for words, is_pos in data if is_pos)
    n_keyword = sum(1 for words, is_pos in data if keyword in words)
    n_keyword_and_pos = sum(1 for words, is_pos in data if is_pos and keyword in words)

    if n_keyword == 0:
        return None  # keyword never appears

    prior = n_pos / total                      # P(Positive)
    likelihood = n_keyword_and_pos / n_pos     # P(keyword | Positive)
    marginal = n_keyword / total               # P(keyword)
    posterior = likelihood * prior / marginal  # P(Positive | keyword)

    return {"keyword": keyword, "prior": prior, "likelihood": likelihood,
            "marginal": marginal, "posterior": posterior,
            "check": n_keyword_and_pos / n_keyword}

In [26]:
positive_keywords = ["excellent", "wonderful", "amazing"]
negative_keywords = ["terrible", "waste", "boring"]

print(f"{'keyword':<12}{'prior':>8}{'likelihood':>12}{'marginal':>10}{'posterior':>11}{'check':>8}")
for kw in positive_keywords + negative_keywords:
    r = bayes_for_keyword(data, kw)
    if r:
        print(f"{kw:<12}{r['prior']:>8.4f}{r['likelihood']:>12.4f}{r['marginal']:>10.4f}{r['posterior']:>11.4f}{r['check']:>8.4f}")

keyword        prior  likelihood  marginal  posterior   check
excellent     0.5000      0.1145    0.0709     0.8074  0.8074
wonderful     0.5000      0.0902    0.0555     0.8120  0.8120
amazing       0.5000      0.0671    0.0431     0.7783  0.7783
terrible      0.5000      0.0152    0.0539     0.1414  0.1414
waste         0.5000      0.0070    0.0506     0.0687  0.0687
boring        0.5000      0.0236    0.0610     0.1936  0.1936


In [19]:
def show_keyword_block(result):
    """Print one small table for a single keyword."""
    print(f"Keyword: {result['keyword']}")
    print(f"  Prior      P(Positive)           = {result['prior']:.4f}")
    print(f"  Likelihood P(keyword | Positive) = {result['likelihood']:.4f}")
    print(f"  Marginal   P(keyword)            = {result['marginal']:.4f}")
    print(f"  Posterior  P(Positive | keyword) = {result['posterior']:.4f}")
    print()

for kw in positive_keywords + negative_keywords:
    show_keyword_block(bayes_for_keyword(data, kw))

Keyword: excellent
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.1145
  Marginal   P(keyword)            = 0.0709
  Posterior  P(Positive | keyword) = 0.8074

Keyword: wonderful
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0902
  Marginal   P(keyword)            = 0.0555
  Posterior  P(Positive | keyword) = 0.8120

Keyword: amazing
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0671
  Marginal   P(keyword)            = 0.0431
  Posterior  P(Positive | keyword) = 0.7783

Keyword: terrible
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0152
  Marginal   P(keyword)            = 0.0539
  Posterior  P(Positive | keyword) = 0.1414

Keyword: waste
  Prior      P(Positive)           = 0.5000
  Likelihood P(keyword | Positive) = 0.0070
  Marginal   P(keyword)            = 0.0506
  Posterior  P(Positive | keyword) = 0.0687

Keyword: boring
  Prior    

In [27]:
def show_keyword_counts(data, keywords):
    """Display the review counts behind each probability."""
    total = len(data)
    positive = sum(is_positive for _, is_positive in data)

    print(f"Total reviews: {total}")
    print(f"Positive reviews: {positive}\n")
    print(f"{'Keyword':<12} {'Contains':>10} {'Positive + word':>17}")

    for keyword in keywords:
        contains = sum(
            keyword in words
            for words, _ in data
        )
        positive_contains = sum(
            is_positive and keyword in words
            for words, is_positive in data
        )

        print(
            f"{keyword:<12} {contains:>10} "
            f"{positive_contains:>17}"
        )


keywords = positive_keywords + negative_keywords
show_keyword_counts(data, keywords)

Total reviews: 50000
Positive reviews: 25000

Keyword        Contains   Positive + word
excellent          3546              2863
wonderful          2777              2255
amazing            2156              1678
terrible           2695               381
waste              2531               174
boring             3048               590


In [28]:
def show_bayes_example(data, keyword):
    """Show substitution into Bayes' theorem."""
    result = bayes_for_keyword(data, keyword)

    if result is None:
        print(f"No reviews contain '{keyword}'.")
        return

    prior = result["prior"]
    likelihood = result["likelihood"]
    marginal = result["marginal"]
    posterior = result["posterior"]

    print(f"Keyword: {keyword}")
    print("P(Positive | word)")
    print("= P(word | Positive) × P(Positive) / P(word)")
    print(f"= {likelihood:.6f} × {prior:.6f} / {marginal:.6f}")
    print(f"= {posterior:.6f} ({posterior:.2%})")


show_bayes_example(data, "excellent")

Keyword: excellent
P(Positive | word)
= P(word | Positive) × P(Positive) / P(word)
= 0.114520 × 0.500000 / 0.070920
= 0.807389 (80.74%)


### Understanding the counts

A keyword is counted once per review, regardless of how often
it appears in that review.

The prior uses all reviews. The likelihood uses positive reviews.
The marginal measures how frequently the keyword appears across
all reviews. Bayes' theorem combines these to calculate the
probability that a review is positive given that keyword.

This measures an association in this dataset. A keyword alone
does not guarantee the sentiment of an individual review.